In [ ]:
---
title: "R Notebook"
output:
  pdf_document: default
  html_notebook: default
---

This is an R Markdown Notebook. When you execute code within the notebook, the results appear beneath the code.

Try executing this chunk by clicking the Run button within the chunk or by placing your cursor inside it and pressing Cmd+Shift+Enter.

{r}
# 加载必要的R包
library(dior)
library(SingleCellExperiment)

# 指定包含 .rds 文件的目录
input_dir <- "/Users/fernandozeng/Desktop/analysis/b808eac4-2fd5-4676-8f8b-8e3cb9d2b791"
output_dir <- "/Users/fernandozeng/Desktop/analysis/b808eac4-2fd5-4676-8f8b-8e3cb9d2b791/out"

# 获取目录中所有 .rds 文件的列表
rds_files <- list.files(input_dir, pattern = "\\.rds$", full.names = TRUE)

# 创建输出目录（如果不存在）
if (!dir.exists(output_dir)) {
  dir.create(output_dir, recursive = TRUE)
}



Add a new chunk by clicking the Insert Chunk button on the toolbar or by pressing Cmd+Option+I.

{r}
# 循环读取每个 .rds 文件并转换为 .h5 格式
for (rds_file in rds_files) {
  # 读取 .rds 文件
  data <- readRDS(rds_file)
  
  # 构造输出文件名
  h5_file <- file.path(output_dir, paste0(tools::file_path_sans_ext(basename(rds_file)), ".h5"))
  
  # 将数据写入 .h5 文件
  dior::write_h5(data, file = h5_file,)
  
  cat("Converted", rds_file, "to", h5_file, "\n")
}


{r}
library(Seurat)

seurat_object <- readRDS(rds_files[1])
# 假设 seurat_object 是你的 Seurat 对象
# 将 counts 数据提取出来
counts_data <- GetAssayData(seurat_object, slot = "counts")

# 将 counts 数据设置为 data 插槽中的数据
seurat_object <- SetAssayData(seurat_object, slot = "data", new.data = counts_data)

{r}
library(sceasy)
library(reticulate)

# 确保使用正确的 conda 环境
use_condaenv('scbasset')

# 导入 Python 的 loompy 模块（如果需要）
loompy <- reticulate::import('loompy')

# 将 Seurat 对象转换为 AnnData 格式
sceasy::convertFormat(seurat_object, from = "seurat", to = "anndata",
                      outFile = 'srt.h5ad')

